# M4 — Fine-tuning VGG16 (Colab)

Reprise du **meilleur checkpoint** `M4_VGG16_multiclass_best.keras`. Essai 1 : bloc 5 dégelé, LR `1e-5`, 5 epochs max. Même split et même preprocessing que le notebook initial. **Pas de test**.

Les CSV, le dataset décompressé et le checkpoint sont à la racine de `DS/projet`. Les résultats sont enregistrés dans `DS/projet/M4_finetuning/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

tf.keras.utils.set_random_seed(42)
PROJECT_DIR = Path('/content/drive/MyDrive/Apprentissage/DS/projet')
DATASET_ROOT = PROJECT_DIR / 'COVID-19_Radiography_Dataset'
OUTPUT_DIR = PROJECT_DIR / 'M4_finetuning'
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT = PROJECT_DIR / 'M4_VGG16_multiclass_best.keras'
print('GPU :', tf.config.list_physical_devices('GPU'))

In [ ]:
train_df = pd.read_csv(PROJECT_DIR / 'train.csv')
val_df = pd.read_csv(PROJECT_DIR / 'validation.csv')

def drive_filepath(original):
    normalized = str(original).replace('\\', '/')
    marker = 'COVID-19_Radiography_Dataset/'
    return str(DATASET_ROOT / normalized.split(marker, 1)[1])

train_df['filepath'] = train_df['filepath'].map(drive_filepath)
val_df['filepath'] = val_df['filepath'].map(drive_filepath)
print('Train :', len(train_df), 'Validation :', len(val_df))

In [ ]:
classes = sorted(train_df['class'].dropna().unique().tolist())
train_df['class'] = train_df['class'].astype(str)
val_df['class'] = val_df['class'].astype(str)
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
train_generator = datagen.flow_from_dataframe(train_df, x_col='filepath', y_col='class', classes=classes,
    target_size=(224,224), color_mode='rgb', class_mode='categorical', batch_size=32, shuffle=True, seed=42)
val_generator = datagen.flow_from_dataframe(val_df, x_col='filepath', y_col='class', classes=classes,
    target_size=(224,224), color_mode='rgb', class_mode='categorical', batch_size=32, shuffle=False)
print('Classes :', val_generator.class_indices)

In [ ]:
# Configuration de l'essai : repartir du checkpoint initial à chaque nouvel essai.
EXPERIMENT_NAME = 'M4_FT_B5_LR1e-5'
LEARNING_RATE = 1e-5
EPOCHS = 5

model = load_model(CHECKPOINT)
base_model = model.layers[0]  # VGG16 dans Sequential
base_model.trainable = True
for layer in base_model.layers:
    layer.trainable = layer.name.startswith('block5_')
print('Couches VGG16 entraînables :', [l.name for l in base_model.layers if l.trainable])

model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='categorical_crossentropy', metrics=['accuracy'])
BEST_PATH = OUTPUT_DIR / f'{EXPERIMENT_NAME}_best.keras'
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint(str(BEST_PATH), monitor='val_loss', save_best_only=True),
    CSVLogger(str(OUTPUT_DIR / f'{EXPERIMENT_NAME}_history.csv')),
]

In [ ]:
history = model.fit(train_generator, validation_data=val_generator,
                    epochs=EPOCHS, callbacks=callbacks)
print('Meilleur checkpoint :', BEST_PATH)

In [ ]:
best_model = load_model(BEST_PATH)
val_generator.reset()
y_prob = best_model.predict(val_generator, verbose=1)
y_true = val_generator.classes
y_pred = np.argmax(y_prob, axis=1)
report = classification_report(y_true, y_pred, target_names=classes, output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(OUTPUT_DIR / f'{EXPERIMENT_NAME}_classification_report.csv')
pd.DataFrame({'y_true': y_true, 'y_pred': y_pred}).to_csv(OUTPUT_DIR / f'{EXPERIMENT_NAME}_predictions.csv', index=False)
print(classification_report(y_true, y_pred, target_names=classes, digits=4, zero_division=0))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=classes).plot(cmap='Blues', xticks_rotation=35)
plt.title(EXPERIMENT_NAME + ' — validation')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'{EXPERIMENT_NAME}_confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()
print('Recall COVID :', report['COVID']['recall'], 'F1 COVID :', report['COVID']['f1-score'])

In [ ]:
hist = pd.read_csv(OUTPUT_DIR / f'{EXPERIMENT_NAME}_history.csv')
for metric in ('loss', 'accuracy'):
    plt.figure(figsize=(7, 4))
    plt.plot(hist[metric], label='Train')
    plt.plot(hist['val_' + metric], label='Validation')
    plt.title(f'{EXPERIMENT_NAME} — {metric}')
    plt.xlabel('Epoch')
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{EXPERIMENT_NAME}_{metric}.png', dpi=160)
    plt.show()